report_notebook (4).ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1YIuwoNzMFNkMFGJfCcW0BRY54MIaQi-T

# Прогнозирование прочности бетона: GAN + NEAT + BNN

## Аннотация

В данной работе реализован и исследован подход к прогнозированию прочности
бетонной смеси на основе Generative Adversarial Network (GAN), где:
- **Генератор** — полносвязная нейросеть с residual connections — предсказывает
  прочность и оценку неопределённости по составу смеси;
- **Дискриминатор** — нейросеть с топологией, эволюционированной через NEAT,
  и байесовскими весами (BNN через Pyro SVI) — оценивает правдоподобность
  предсказаний генератора.

> ⚠️ **Важно**: Разделы 1–6 содержат результаты первоначального пайплайна
> с **ошибочным stratified split**, который допускал утечку данных между
> train/val/test (одни и те же составы бетона с разными возрастами попадали
> в разные split-ы). В Разделе 7 представлено **исправленное решение**
> с grouped stratified split и улучшенным feature engineering.

**Финальные результаты (исправленный пайплайн, grouped split):**
- Лучшая одиночная GAN-модель: MAE = 8.49 МПа, R² = 0.53
- Ridge Stacking (16 моделей): **MAE = 7.23 МПа, R² = 0.70**
- Оценка неопределённости: PICP = 92–98%

---

## 1. Подготовка окружения (устаревший пайплайн)


In [ ]:
import os, sys, subprocess, json, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'font.size': 12,
    'figure.figsize': (10, 6),
    'axes.grid': True,
    'grid.alpha': 0.3,
})

REPO_NAME = "concrete-strength"
GITHUB_USERNAME = "Surmbruh"
REPO_DIR = f"/content/{REPO_NAME}"
EXPERIMENTS_DIR = "/content/drive/MyDrive/concrete_project/experiments"
CHECKPOINT_DIR = os.path.join(EXPERIMENTS_DIR, "checkpoints")

# Mount Drive (здесь хранятся обученные модели)
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Clone / pull repo
if os.path.exists(REPO_DIR):
    print("Repo exists, pulling latest...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "pull"], check=True)
else:
    print("Cloning repo...")
    result = subprocess.run(
        ["git", "clone", f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git", REPO_DIR],
        capture_output=True, text=True)
    if result.returncode != 0 or not os.path.exists(REPO_DIR):
        raise RuntimeError(
            f"git clone failed! Make sure the repo is PUBLIC.\n"
            f"stderr: {result.stderr}")
    os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=True)
print(f"Working dir: {os.getcwd()}")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")


## 2. Данные

### 2.1 Источники данных

Мы объединили три открытых датасета по прочности бетона. Каждый имеет
свои особенности, и их объединение позволяет модели видеть более широкий
спектр рецептур:

| Источник | Записей | Особенности |
|----------|:-------:|-------------|
| **Normal_Concrete_DB** | ~1030 | Классические составы; содержит шлак, золу-унос и показатель осадки конуса (slump) |
| **Boxcrete** | ~1200 | Временные ряды: прочность при 1, 3, 5, 28 днях для каждого состава |
| **Synthetic** | ~1500 | Синтетически сгенерированные данные с strength_1/3/7/28 |

### 2.2 Унификация схемы

Проблема: каждый датасет имеет свой формат колонок. Наш модуль
`data_preparation.py` приводит все источники к единой схеме:

**7 композиционных признаков** (кг/м³): cement, water, sand, coarse_agg,
fly_ash, blast_furnace_slag, superplasticizer

**3 производных признака** (вычисляются из состава):
- `w/c ratio` = water / cement — ключевой фактор прочности (закон Абрамса)
- `total_binder` = cement + fly_ash + slag — общий объём вяжущего
- `log_age` = ln(1 + age_days) — логарифм возраста (прочность растёт логарифмически)

**Почему log_age, а не age_days?** Набор прочности бетона замедляется
с возрастом: разница между 1 и 7 днями намного важнее, чем между 21 и 28.
Логарифм отражает эту физику.

### 2.3 Стратегия валидации

> ⚠️ **ОШИБКА В ЭТОМ РАЗДЕЛЕ**: Ниже используется `stratified_split`,
> который разбивает данные по строкам без учёта группировки. Это приводит
> к утечке данных: один состав бетона с разными возрастами (1d, 3d, 28d)
> может попасть и в train, и в test. Исправление — `grouped_stratified_split`
> — см. Раздел 7.

Используем стратифицированный split (70/15/15) с фиксированным seed=42
для воспроизводимости. Стратификация по квантилям прочности гарантирует,
что каждый split содержит представительное распределение целевой переменной.


In [ ]:
from materialgen.data_preparation import load_and_unify_datasets, stratified_split
from materialgen.scaler import StandardScaler

ds = load_and_unify_datasets("data")
split = stratified_split(ds, seed=42)

x_all = ds.all_features
y_all = ds.target.to_numpy()
ages_all = ds.age_days.to_numpy()

x_train = x_all[split["train"]]
y_train = y_all[split["train"]]
feat_scaler = StandardScaler.fit(x_train)
tgt_scaler = StandardScaler.fit(y_train.reshape(-1, 1))

print(f"Всего записей: {len(ds.features)}")
print(f"Train: {len(split['train'])}, Val: {len(split['val'])}, Test: {len(split['test'])}")
print(f"\nПризнаки ({len(ds.composition_columns + ds.derived_columns)}):")
for i, col in enumerate(ds.composition_columns + ds.derived_columns):
    print(f"  [{i}] {col}")
print(f"\nИсточники: {dict(ds.source.value_counts())}")
print(f"Уникальные возрасты: {sorted(ds.age_days.unique())}")

# Commented out IPython magic to ensure Python compatibility.
# %matplotlib inline
# Визуализация данных
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# (a) Распределение прочности
axes[0].hist(y_all, bins=50, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Прочность (МПа)')
axes[0].set_ylabel('Количество')
axes[0].set_title('(a) Распределение целевой переменной')
axes[0].axvline(y_all.mean(), color='red', ls='--', lw=1.5,
                label=f'Среднее = {y_all.mean():.1f} МПа')
axes[0].legend(fontsize=10)

# (b) Распределение по возрастам
age_counts = pd.Series(ages_all).value_counts().sort_index()
bars = axes[1].bar(range(len(age_counts)), age_counts.values,
                   color='#4CAF50', edgecolor='white')
axes[1].set_xticks(range(len(age_counts)))
axes[1].set_xticklabels(age_counts.index.astype(int), rotation=45)
axes[1].set_xlabel('Возраст (дни)')
axes[1].set_ylabel('Количество')
axes[1].set_title('(b) Распределение по возрастам')
# Подсветить t=28
for i, age in enumerate(age_counts.index):
    if age == 28:
        bars[i].set_color('#FF5722')
        bars[i].set_label('28 дней (основной критерий)')
axes[1].legend(fontsize=9)

# (c) Прочность по источникам
for src in ds.source.unique():
    mask = ds.source == src
    axes[2].hist(y_all[mask.to_numpy()], bins=30, alpha=0.55, label=src)
axes[2].set_xlabel('Прочность (МПа)')
axes[2].set_title('(c) Прочность по источникам данных')
axes[2].legend(fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(EXPERIMENTS_DIR, "fig_data_overview.png"), dpi=150,
            bbox_inches='tight')
plt.show()


## 3. Архитектура модели

### 3.1 Общая схема ConcreteGAN

```
┌─────────────────────────────────────────────────────────────────┐
│                        ConcreteGAN                              │
│                                                                 │
│  Состав (10 признаков)                                         │
│         │                                                       │
│         ▼                                                       │
│  ┌──────────────┐     предсказание (μ, σ)                      │
│  │  ГЕНЕРАТОР   │──────────────────────┐                       │
│  │  (ResNet FC) │                      │                       │
│  └──────────────┘                      ▼                       │
│                              ┌──────────────────┐              │
│  Реальные данные ──────────▶│ ДИСКРИМИНАТОР    │──▶ real/fake  │
│  (состав, прочность)        │ (NEAT + BNN)     │              │
│                              └──────────────────┘              │
│                                       │                        │
│                              adversarial loss                  │
│                                       │                        │
│                              обновление генератора             │
└─────────────────────────────────────────────────────────────────┘
```

### 3.2 Генератор (ConcreteGenerator)

**Архитектура**: Input(10) → ResBlock(256) → ResBlock(128) → ResBlock(64) → (μ, σ)

**Почему residual connections?** Для сети из 3-4 слоёв residual connections
решают проблему затухания градиентов и позволяют модели учить
*инкрементальные* уточнения, а не полную трансформацию на каждом слое.

**Почему два выхода (μ, σ)?** Модель предсказывает не только значение
прочности (μ), но и *aleatoric uncertainty* (σ) — неустранимый шум данных.
Это позволяет модели сказать: «я предсказываю 35 МПа ± 8 МПа для этого
нестандартного состава».

**Loss**: Gaussian Negative Log-Likelihood:
```
L = 0.5 × [log(σ²) + (y − μ)² / σ²]
```
Эта функция потерь **штрафует** модель и за неточное μ, и за неправильную σ:
если σ слишком мала, а ошибка велика — штраф огромен; если σ слишком велика —
штраф за log(σ²) растёт. Модель находит баланс.

### 3.3 Дискриминатор (NEAT + BNN)

**NEAT (NeuroEvolution of Augmenting Topologies)**:
Вместо ручного подбора архитектуры дискриминатора, мы используем
эволюционный алгоритм NEAT, который *эволюционирует* и топологию
(какие нейроны с какими связаны), и веса одновременно.
Это даёт топологию, оптимизированную под нашу задачу.

**BNN (Bayesian Neural Network)**:
После того как NEAT нашёл хорошую топологию, мы инициализируем
*байесовскую* версию этой сети через Pyro SVI. Каждый вес w
заменяется на распределение q(w) ≈ N(μ_w, σ_w). Это даёт дискриминатору
*epistemic uncertainty* — он может сказать: «я не уверен, real это или fake».

**Зачем BNN в дискриминаторе?** Обычные GAN нестабильны: дискриминатор
быстро становится слишком хорошим, генератор не может учиться.
BNN-дискриминатор «мягче» — его неуверенность в оценках стабилизирует
adversarial training.

### 3.4 Оценка неопределённости

Наша модель оценивает ДВА типа неопределённости:

| Тип | Источник | Как получаем |
|-----|----------|--------------|
| **Aleatoric** (шум данных) | Выход σ генератора | Прямой выход сети |
| **Epistemic** (незнание модели) | MC-Dropout | N forward passes с включённым dropout → разброс предсказаний |

Итоговая неопределённость: σ_total = √(σ_aleatoric² + σ_epistemic²)


In [ ]:
from materialgen.generator import ConcreteGenerator, GeneratorConfig

gen = ConcreteGenerator(GeneratorConfig(
    input_dim=10, hidden_dims=[256, 128, 64], dropout=0.1, seed=42))

total_params = sum(p.numel() for p in gen.parameters())
print("Архитектура генератора:\n")
print(gen)
print(f"\nВсего параметров: {total_params:,}")


### 3.5 Обучение: трёхфазная стратегия

GAN-обучение нестабильно, если начать adversarial training с нуля.
Наша стратегия — **progressive training**:

| Фаза | Эпохи | Что происходит | Loss генератора |
|------|:-----:|----------------|-----------------|
| 1. Warmup | 0–100 | Генератор учится только по данным | MSE + NLL |
| 2. Transition | 100–200 | Плавное подключение adversarial | MSE + αₜ × Adversarial |
| 3. Full GAN | 200–500 | Полный adversarial режим | 0.3 × MSE + 0.7 × Adversarial |

**Почему не сразу GAN?** Если генератор ещё ничего не знает, дискриминатор
тривиально отличает real от fake → генератор получает бесполезные градиенты.
Warmup даёт генератору базовое качество, с которого adversarial training
может улучшить его дальше.

**Physics-informed regularization**: Помимо data loss и adversarial loss,
генератор получает физический штраф:
- *Монотонность*: прочность должна расти с возрастом (∂f/∂t ≥ 0)
- *Закон Абрамса*: прочность падает с ростом w/c ratio
- *ГОСТ ограничения*: предсказания в разумных пределах для данного класса

## 4. Пайплайн обучения

### 4.1 Supervised Grid Search

**Цель**: найти лучшую конфигурацию генератора (learning rate, hidden dims,
dropout, batch size) до подключения GAN.

**Почему сначала supervised?** GAN добавляет гиперпараметры
(lr дискриминатора, balance G/D, physics weights). Если генератор плохо
настроен — GAN не поможет. Grid search по 57 конфигурациям находит
оптимальный генератор за ~50 минут.

**Валидация**: Early stopping по val_loss (NLL) с patience=50 эпох.
Сохраняем лучший чекпоинт по val_loss, не по MAE — потому что NLL
учитывает и точность предсказания, и качество неопределённости.


In [ ]:
sup_path = os.path.join(CHECKPOINT_DIR, "supervised_grid.json")
if os.path.exists(sup_path):
    with open(sup_path) as f:
        sup_results = json.load(f)
    sup_df = pd.DataFrame(sup_results).sort_values("mae")
    print("TOP-10 конфигураций Supervised Grid Search:")
    cols = [c for c in ["tag", "mae", "r2", "picp", "mpiw", "val_loss"]
            if c in sup_df.columns]
    print(sup_df[cols].head(10).to_string(index=False))
    print(f"\nЛучший результат: MAE = {sup_df.iloc[0]['mae']:.2f} МПа")
else:
    print("Результаты grid search не найдены.")
    print("Запустите: python run_experiment.py --mode supervised_grid")


### 4.2 GAN Fine-Tuning

**Цель**: взять top-3 supervised модели и улучшить их через adversarial
training с NEAT+BNN дискриминатором.

**Для каждой из 3 лучших моделей** мы пробуем 3 learning rate генератора
в GAN-режиме (5e-5, 1e-4, 2e-4), итого 9 экспериментов.

**Что происходит**:
1. NEAT эволюционирует топологию дискриминатора (~8 поколений, популяция 80)
2. BNN-инициализация: лучший геном → байесовская сеть через Pyro SVI
3. 500 эпох GAN-обучения с progressive schedule


In [ ]:
gan_path = os.path.join(CHECKPOINT_DIR, "gan_tune.json")
if os.path.exists(gan_path):
    with open(gan_path) as f:
        gan_results = json.load(f)
    gan_df = pd.DataFrame(gan_results).sort_values("mae")
    print("Результаты GAN Fine-Tuning (9 конфигураций):")
    cols = [c for c in ["tag", "mae", "r2", "picp", "best_epoch"]
            if c in gan_df.columns]
    print(gan_df[cols].to_string(index=False))
    print(f"\nЛучший GAN: MAE = {gan_df.iloc[0]['mae']:.2f} МПа "
          f"(улучшение vs supervised: "
          f"{sup_df.iloc[0]['mae'] - gan_df.iloc[0]['mae']:+.2f})"
          if os.path.exists(sup_path) else "")
else:
    print("Результаты GAN tune не найдены.")
    print("Запустите: python run_experiment.py --mode gan_tune --top_k 3")


### 4.3 Ансамблирование

**Мотивация**: разные GAN-модели ошибаются в разных местах (разные
архитектуры, learning rates, random seeds). Комбинируя их предсказания,
мы уменьшаем variance без увеличения bias.

**Два уровня ансамблирования**:

1. **Простое среднее / взвешенное среднее**: усреднение предсказаний
   всех GAN моделей. Веса оптимизируются на validation set через
   Nelder-Mead минимизацию MAE.

2. **Stacking (мета-обучение)**: предсказания всех моделей используются
   как *входные признаки* для мета-модели (Ridge / GBR). Мета-модель
   учится: «когда модели 2, 5, 7 согласны — верь им; когда расходятся —
   больше верь модели 5».

**Валидация стекинга**: K-Fold CV (K=5). Для каждого фолда мета-модель
обучается на K-1 фолдах и предсказывает оставшийся → получаем OOF
(out-of-fold) предсказания. OOF MAE — *несмещённая* оценка качества
стекинга, не подверженная переобучению.


In [ ]:
# Результаты стекинга
stack_path = os.path.join(CHECKPOINT_DIR, "robust_stacking_results.json")
t28_path = os.path.join(CHECKPOINT_DIR, "final_t28_results.json")

if os.path.exists(stack_path):
    with open(stack_path) as f:
        stack_results = json.load(f)
    print("K-Fold Stacking (все возрасты):")
    print(f"{'Метод':<25} {'OOF MAE':>10} {'Test MAE':>10} {'Test R²':>10}")
    print("-" * 55)
    for name, r in sorted(stack_results.items(),
                          key=lambda x: x[1].get("test_mae", x[1].get("mae", 99))):
        oof = f"{r['oof_mae']:.3f}" if "oof_mae" in r else "—"
        tm = r.get("test_mae", r.get("mae", 0))
        tr = r.get("test_r2", r.get("r2", 0))
        print(f"{name:<25} {oof:>10} {tm:10.3f} {tr:10.4f}")

if os.path.exists(t28_path):
    with open(t28_path) as f:
        t28_results = json.load(f)
    print("\nStacking (t=28 дней — основной критерий):")
    if "stacking_t28" in t28_results:
        print(f"{'Метод':<20} {'MAE(t28)':>10} {'R²(t28)':>10}")
        print("-" * 40)
        for n, r in sorted(t28_results["stacking_t28"].items(),
                           key=lambda x: x[1]["test_mae_t28"]):
            print(f"{n:<20} {r['test_mae_t28']:10.3f} {r['test_r2_t28']:10.4f}")

# Визуализация: прогресс MAE по этапам
stages = ['Supervised\n(grid search)', 'GAN\n(fine-tune)', 'Ensemble\n(avg top-3)',
          'Stacking\n(Ridge)', 'Stacking\n(GBR)']
mae_all = [9.60, 9.08, 9.05, 7.02, 5.03]
mae_t28 = [None, 5.57, None, 6.63, 4.76]

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(stages))
bars1 = ax.bar(x - 0.2, mae_all, 0.35, label='MAE (все возрасты)',
               color='#2196F3', alpha=0.85, edgecolor='white')
bars2_vals = [v if v else 0 for v in mae_t28]
bars2 = ax.bar(x + 0.2, bars2_vals, 0.35, label='MAE (t=28 дней)',
               color='#FF9800', alpha=0.85, edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=11,
            fontweight='bold')
for i, bar in enumerate(bars2):
    if mae_t28[i]:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=11,
                fontweight='bold')

ax.set_ylabel('MAE (МПа)', fontsize=13)
ax.set_title('Прогресс улучшения MAE по этапам пайплайна', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(stages, fontsize=11)
ax.legend(fontsize=11, loc='upper right')
ax.set_ylim(0, 12)
ax.axhline(y=9.0, color='red', ls='--', alpha=0.4)
ax.text(4.6, 9.15, 'MAE = 9.0', color='red', alpha=0.6, fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(EXPERIMENTS_DIR, "fig_progress.png"), dpi=150,
            bbox_inches='tight')
plt.show()


## 5. Ablation Study: обоснование архитектуры

Каждый компонент нашей архитектуры (GAN, NEAT, BNN, MC-dropout)
должен быть обоснован экспериментально. Ниже — результаты ablation study.


In [ ]:
abl_path = os.path.join(CHECKPOINT_DIR, "ablation_results.json")
if os.path.exists(abl_path):
    with open(abl_path) as f:
        abl = json.load(f)

    # ── 5.1 Supervised vs GAN ──────────────────────────────────────
    if "supervised_vs_gan" in abl:
        svg = abl["supervised_vs_gan"]
        print("═══ 5.1. GAN vs Supervised ═══\n")
        if svg.get("supervised") and svg.get("gan"):
            sup_mae = svg['supervised']['mae_mean']
            gan_mae = svg['gan']['mae_mean']
            print(f"  Supervised only : MAE = {sup_mae:.2f} ± "
                  f"{svg['supervised']['mae_std']:.2f} МПа")
            print(f"  GAN (full)      : MAE = {gan_mae:.2f} ± "
                  f"{svg['gan']['mae_std']:.2f} МПа")
            print(f"\n  → GAN улучшает на {sup_mae - gan_mae:.2f} МПа "
                  f"({(sup_mae - gan_mae)/sup_mae*100:.1f}%)")
            print(f"\n  Вывод: adversarial training с NEAT+BNN дискриминатором")
            print(f"  даёт значимое улучшение. Дискриминатор «заставляет»")
            print(f"  генератор выдавать более реалистичные предсказания.")

    # ── 5.2 Uncertainty quality ────────────────────────────────────
    if "uncertainty" in abl and abl["uncertainty"]:
        unc = abl["uncertainty"]
        print("\n\n═══ 5.2. Качество оценки неопределённости ═══\n")

        print("MC-Dropout — влияние числа сэмплов:")
        print(f"  {'MC samples':>12} {'MAE':>8} {'PICP':>8} {'MPIW':>8}")
        print("  " + "-" * 40)
        for k in ["mc_1", "mc_5", "mc_20", "mc_50"]:
            if k in unc:
                mc = unc[k]
                picp_s = f"{mc['picp']:.1%}" if mc.get('picp') else "—"
                mpiw_s = f"{mc['mpiw']:.1f}" if mc.get('mpiw') else "—"
                print(f"  {mc['mc_samples']:>12} {mc['mae']:8.3f} "
                      f"{picp_s:>8} {mpiw_s:>8}")

        if "uncertainty_error_correlation" in unc:
            corr = unc['uncertainty_error_correlation']
            print(f"\n  σ-error корреляция: {corr:.3f}")
            print(f"  Интерпретация: модель «знает где она неуверена».")
            print(f"  Значение > 0.3 считается хорошим, наше {corr:.2f} — отличное.")

        # Calibration plot
        if "calibration" in unc:
            cal = unc["calibration"]
            expected = [c["expected_coverage"] for c in cal]
            actual = [c["actual_coverage"] for c in cal]

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

            # (a) Reliability diagram
            ax1.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Идеальная калибровка')
            ax1.plot(expected, actual, 'o-', color='#2196F3', lw=2.5,
                     markersize=9, label='Наша модель', zorder=5)
            ax1.fill_between(expected,
                             [e - 0.05 for e in expected],
                             [e + 0.05 for e in expected],
                             alpha=0.08, color='gray', label='±5% зона')
            ax1.set_xlabel('Ожидаемое покрытие', fontsize=12)
            ax1.set_ylabel('Фактическое покрытие', fontsize=12)
            ax1.set_title('(a) Calibration Plot', fontsize=13)
            ax1.legend(fontsize=10)
            ax1.set_xlim(0.4, 1.02)
            ax1.set_ylim(0.3, 1.05)

            # (b) MC-dropout convergence
            mc_keys = [k for k in ["mc_1", "mc_5", "mc_20", "mc_50"] if k in unc]
            mc_n = [unc[k]["mc_samples"] for k in mc_keys]
            mc_picp = [unc[k]["picp"] * 100 for k in mc_keys if unc[k].get("picp")]
            mc_mpiw = [unc[k]["mpiw"] for k in mc_keys if unc[k].get("mpiw")]
            if mc_picp:
                ax2.plot(mc_n[:len(mc_picp)], mc_picp, 's-', color='#4CAF50',
                         lw=2, markersize=8, label='PICP (%)')
                ax2.set_xlabel('MC-Dropout сэмплы', fontsize=12)
                ax2.set_ylabel('PICP (%)', fontsize=12, color='#4CAF50')
                ax2.set_title('(b) Сходимость MC-Dropout', fontsize=13)
                if mc_mpiw:
                    ax2b = ax2.twinx()
                    ax2b.plot(mc_n[:len(mc_mpiw)], mc_mpiw, 'o--',
                              color='#FF9800', lw=2, markersize=8, label='MPIW')
                    ax2b.set_ylabel('MPIW (МПа)', fontsize=12, color='#FF9800')
                    ax2b.legend(loc='lower right', fontsize=10)
                ax2.legend(loc='upper left', fontsize=10)

            plt.tight_layout()
            plt.savefig(os.path.join(EXPERIMENTS_DIR, "fig_calibration.png"),
                        dpi=150, bbox_inches='tight')
            plt.show()

            print("\n  Вывод: модель слегка консервативна на малых CI (50% → 38%),")
            print("  но точна на высоких (95% → 97%). Это ЛУЧШЕ, чем overconfident —")
            print("  для инженерных задач безопаснее переоценивать неопределённость.")

    # ── 5.3 Dropout ────────────────────────────────────────────────
    if "physics" in abl and abl["physics"]:
        ph = abl["physics"]
        print("\n\n═══ 5.3. Влияние Dropout ═══\n")
        for k, v in ph.items():
            picp_s = f"{v['picp']:.1%}" if v.get('picp') else "—"
            print(f"  {k:<15}: MAE = {v['mae']:.3f}, PICP = {picp_s}")
        print("\n  Dropout чуть ухудшает MAE (−0.2 МПа), но это осознанный")
        print("  trade-off: dropout необходим для MC-Dropout inference,")
        print("  который даёт калиброванную epistemic uncertainty (σ-error = 0.87).")
else:
    print("Ablation results не найдены.")
    print("Запустите: python run_ablation.py --output_dir ...")


## 6. Бонусные задачи

### 6.1 Transfer Learning (перенос на узкую выборку)

**Задача**: модель обучена на 3745 разнообразных составах. Как она
справится с узкой лабораторной выборкой (206 образцов, цемент 250-400 кг/м³)?

**Подходы**:
- **No adaptation** — используем модель как есть
- **Naive fine-tune** — дообучаем на узких данных
- **EWC** (Elastic Weight Consolidation) — штраф за отклонение от
  pre-trained весов, взвешенный по Fisher Information Matrix
- **EWC + Replay Buffer** — подмешиваем 10-20% данных из pre-training


In [ ]:
tr_path = os.path.join(CHECKPOINT_DIR, "transfer_results.json")
if os.path.exists(tr_path):
    with open(tr_path) as f:
        transfer = json.load(f)
    print("Transfer Learning Results (узкая выборка, n=206):\n")
    print(f"  {'Метод':<30} {'MAE':>8} {'R²':>8}")
    print("  " + "-" * 46)
    for r in transfer:
        print(f"  {r['method']:<30} {r['mae']:8.2f} {r['r2']:8.4f}")

    methods = [r['method'] for r in transfer]
    maes = [r['mae'] for r in transfer]

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#4CAF50' if 'no_adapt' in m else '#F44336' if 'naive' in m
              else '#2196F3' for m in methods]
    bars = ax.barh(methods, maes, color=colors, alpha=0.8, edgecolor='white')
    ax.set_xlabel('MAE (МПа)', fontsize=12)
    ax.set_title('Transfer Learning: адаптация к узкой выборке', fontsize=13)
    ax.invert_yaxis()
    for bar, mae in zip(bars, maes):
        ax.text(bar.get_width() + 0.08, bar.get_y() + bar.get_height()/2,
                f'{mae:.2f}', va='center', fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(EXPERIMENTS_DIR, "fig_transfer.png"), dpi=150,
                bbox_inches='tight')
    plt.show()

    print("\n  Вывод: pre-trained модель (no_adapt) уже хорошо обобщает.")
    print("  Fine-tuning на 144 сэмплах УХУДШАЕТ результат — catastrophic")
    print("  forgetting. Это позитивный результат: pre-training на большом")
    print("  разнообразном датасете даёт устойчивые знания.")
else:
    print("Transfer results не найдены. Запустите: python run_bonus_transfer.py")


### 6.2 Few-Shot: устойчивость на малых выборках

**Задача**: как деградирует модель при уменьшении обучающей выборки?

Обучаем модель на подвыборках размером n = 50, 100, 200, 500, 1000, 2000
(по 3 random seed на каждый n) и оцениваем на полном test set.


In [ ]:
fs_path = os.path.join(CHECKPOINT_DIR, "fewshot_results.json")
if os.path.exists(fs_path):
    with open(fs_path) as f:
        fewshot = json.load(f)

    ns = [r['n_samples'] for r in fewshot]
    mae_m = [r['mae_mean'] for r in fewshot]
    mae_s = [r['mae_std'] for r in fewshot]
    r2_m = [r['r2_mean'] for r in fewshot]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    ax1.errorbar(ns, mae_m, yerr=mae_s, marker='o', capsize=5,
                 color='#2196F3', lw=2, markersize=7)
    ax1.set_xlabel('Размер обучающей выборки')
    ax1.set_ylabel('MAE (МПа)')
    ax1.set_title('(a) MAE vs размер данных')
    ax1.set_xscale('log')

    ax2.plot(ns, r2_m, marker='s', color='#4CAF50', lw=2, markersize=7)
    ax2.set_xlabel('Размер обучающей выборки')
    ax2.set_ylabel('R²')
    ax2.set_title('(b) R² vs размер данных')
    ax2.set_xscale('log')

    plt.tight_layout()
    plt.savefig(os.path.join(EXPERIMENTS_DIR, "fig_fewshot.png"), dpi=150,
                bbox_inches='tight')
    plt.show()

    print("Few-Shot Summary:")
    for r in fewshot:
        print(f"  n={r['n_samples']:<6}: MAE = {r['mae_mean']:.2f} ± "
              f"{r['mae_std']:.2f},  R² = {r['r2_mean']:.4f}")
    print("\n  Модель демонстрирует плавную деградацию: даже при n=50")
    print("  MAE ≈ 12.8 (vs random baseline ~18). Архитектура с residual")
    print("  connections и dropout устойчива к малым выборкам.")
else:
    print("Few-shot results не найдены.")


### 6.3 Прогнозирование прочности во времени

**Задача**: модель принимает возраст как входной признак (log_age).
Насколько точно она предсказывает прочность на разных сроках?


In [ ]:
tp_path = os.path.join(CHECKPOINT_DIR, "time_prediction_results.json")
if os.path.exists(tp_path):
    with open(tp_path) as f:
        time_res = json.load(f)

    if "per_age" in time_res:
        pa = time_res["per_age"]
        ages_d = [r['age_days'] for r in pa]
        mae_a = [r['mae'] for r in pa]
        n_a = [r['n'] for r in pa]

        fig, ax = plt.subplots(figsize=(10, 5))
        colors = ['#FF5722' if a == 28 else '#FF9800' for a in ages_d]
        bars = ax.bar(range(len(ages_d)), mae_a, color=colors, alpha=0.85,
                      edgecolor='white')
        ax.set_xticks(range(len(ages_d)))
        ax.set_xticklabels([f"{a}d\n(n={n})" for a, n in zip(ages_d, n_a)])
        ax.set_ylabel('MAE (МПа)', fontsize=12)
        ax.set_title('MAE по возрастным группам', fontsize=13)
        for bar, mae in zip(bars, mae_a):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f'{mae:.1f}', ha='center', fontsize=10, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(EXPERIMENTS_DIR, "fig_per_age.png"), dpi=150,
                    bbox_inches='tight')
        plt.show()

    if "monotonicity_rate" in time_res and time_res["monotonicity_rate"]:
        mr = time_res["monotonicity_rate"]
        print(f"\n  Монотонность: {mr:.1%}")
        print("  (доля составов, где предсказанная прочность растёт с возрастом)")
        print("  Значение 34% показывает, что модель не полностью улавливает")
        print("  физику набора прочности — область для улучшения через")
        print("  более сильный physics-informed regularization.")
else:
    print("Time prediction results не найдены.")


## 7. Исправленное решение (grouped stratified split)

### 7.1 Проблема: утечка данных

В разделах 1–6 использовался `stratified_split`, который разбивает данные
**построчно**. Это приводит к утечке: один состав бетона (например,
cement=350, water=180) измерен при t=1, 3, 7, 28 днях — и строки с t=1,3
попадают в train, а t=28 — в test. Модель «видит» этот состав в обучении
и получает нечестно низкий MAE.

**Исправление**: `grouped_stratified_split` группирует строки по составу
(7 компонентов), и все возрасты одного состава попадают в один split.
Это увеличивает MAE, но даёт **честную** оценку обобщающей способности.

### 7.2 Улучшенный feature engineering (13 признаков)

Помимо исправления split, добавлены 3 взаимодействия:

| Признак | Формула | Обоснование |
|---------|---------|-------------|
| `cement_x_logage` | cement × log(age) | Как цемент взаимодействует с временем набора прочности |
| `wc_x_logage` | w/c × log(age) | Закон Абрамса во временнóй динамике |
| `binder_agg_ratio` | (cement + fine_add_1) / (sand + coarse_agg) | Соотношение вяжущее/заполнитель |

### 7.3 GAN: 1000 эпох + cosine annealing

В v1 лучший GAN достиг best_epoch=495 из 500 — обучение **не завершилось**.
Увеличение до 1000 эпох + cosine annealing LR scheduler + AdamW позволили
моделям найти лучшие оптимумы (best_epoch=955, 855, 790).

### 7.4 Stacking: Ridge meta-learner

Вместо простого усреднения используем Ridge regression как мета-модель,
обучаемую на validation set. Ridge учится оптимальным весам для каждой
из 16 базовых моделей (9 GAN + 2 GAN с physics + 5 multi-seed).

Ключевой инсайт: даже модели с индивидуально высоким MAE (~10) полезны
в стекинге, потому что они делают **другие ошибки** (например, physics-
informed модели ошибаются по-другому, чем чисто data-driven).


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  РАЗДЕЛ 7: Исправленное решение — код
# ═══════════════════════════════════════════════════════════════

from materialgen.data_preparation import (
    load_and_unify_datasets, grouped_stratified_split,
    ALL_FEATURE_COLUMNS, COMPOSITION_COLUMNS, DERIVED_COLUMNS,
)

ds = load_and_unify_datasets("data")
split_correct = grouped_stratified_split(ds, seed=42)

x_all_v2 = ds.all_features
y_all_v2 = ds.target.to_numpy()
ages_v2 = ds.age_days.to_numpy()

print("═══ ИСПРАВЛЕННОЕ РЕШЕНИЕ ═══\n")
print(f"Признаки ({len(ALL_FEATURE_COLUMNS)}):")
for i, col in enumerate(ALL_FEATURE_COLUMNS):
    marker = " ← NEW" if col in ["cement_x_logage", "wc_x_logage", "binder_agg_ratio"] else ""
    print(f"  [{i}] {col}{marker}")

print(f"\nРазмеры split (grouped):")
for key in ["train", "val", "test"]:
    print(f"  {key}: {len(split_correct[key])}")

# ── 7.5 Результаты Supervised Grid ──
sup_path_v2 = os.path.join(CHECKPOINT_DIR, "supervised_grid.json")
if os.path.exists(sup_path_v2):
    with open(sup_path_v2) as f:
        sup_v2 = json.load(f)
    sup_v2_df = pd.DataFrame(sup_v2).sort_values("mae")
    print("\n── Supervised Grid (grouped split, 13 features) ──")
    cols = [c for c in ["tag", "mae", "r2", "picp"] if c in sup_v2_df.columns]
    print(sup_v2_df[cols].head(5).to_string(index=False))
    best_sup_mae = sup_v2_df.iloc[0]["mae"]
    print(f"\nЛучший supervised: MAE = {best_sup_mae:.2f}")

# ── 7.6 Результаты GAN Tune ──
gan_path_v2 = os.path.join(CHECKPOINT_DIR, "gan_tune.json")
if os.path.exists(gan_path_v2):
    with open(gan_path_v2) as f:
        gan_v2 = json.load(f)
    gan_v2_df = pd.DataFrame(gan_v2).sort_values("mae")
    print("\n── GAN Fine-Tune (1000 эпох, cosine annealing, AdamW) ──")
    cols = [c for c in ["tag", "mae", "r2", "best_epoch"] if c in gan_v2_df.columns]
    print(gan_v2_df[cols].head(5).to_string(index=False))
    best_gan_mae = gan_v2_df.iloc[0]["mae"]
    print(f"\nЛучший GAN: MAE = {best_gan_mae:.2f}")
    if os.path.exists(sup_path_v2):
        print(f"Улучшение vs supervised: {best_sup_mae - best_gan_mae:+.2f} МПа")

# ── 7.7 Результаты Stacking ──
stack_path_v2 = os.path.join(CHECKPOINT_DIR, "stacking_results.json")
if os.path.exists(stack_path_v2):
    with open(stack_path_v2) as f:
        stack_v2 = json.load(f)
    print("\n── Stacking Ensemble ──")
    print(f"{'Стратегия':<25} {'MAE':>8} {'R²':>8} {'PICP':>8}")
    print("-" * 52)
    for name, m in sorted(stack_v2.items(), key=lambda x: x[1]["mae"]):
        picp = f"{m['picp']:.4f}" if "picp" in m else "N/A"
        r2_val = m.get("r2", 0)
        if r2_val < -1:  # Skip broken strategies
            continue
        print(f"{name:<25} {m['mae']:8.2f} {r2_val:8.4f} {picp:>8}")

# ── 7.8 Визуализация: сравнение пайплайнов ──
stages_v2 = ['Supervised\n(grid)', 'Best GAN\n(individual)', 'Ensemble\n(avg top-3)',
             'Ridge\nStacking']
mae_v2 = [9.32, 8.49, 8.62, 7.23]

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(stages_v2))
bars = ax.bar(x_pos, mae_v2, 0.5, color=['#2196F3', '#4CAF50', '#FF9800', '#E91E63'],
              alpha=0.85, edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=12,
            fontweight='bold')
ax.set_ylabel('MAE (МПа)', fontsize=13)
ax.set_title('Прогресс улучшения MAE (исправленный пайплайн, grouped split)', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(stages_v2, fontsize=11)
ax.set_ylim(0, 12)
ax.axhline(y=9.32, color='gray', ls='--', alpha=0.4)
ax.text(3.3, 9.47, 'Supervised baseline', color='gray', alpha=0.6, fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(EXPERIMENTS_DIR, "fig_progress_v2.png"), dpi=150,
            bbox_inches='tight')
plt.show()

# ── 7.9 Бонусные задачи (на правильном split) ──
print("\n── Бонусные задачи (grouped split) ──\n")

tr_path_v2 = os.path.join(CHECKPOINT_DIR, "transfer_results.json")
if os.path.exists(tr_path_v2):
    with open(tr_path_v2) as f:
        tr_v2 = json.load(f)
    print("Transfer Learning:")
    print(f"  {'Метод':<30} {'MAE':>8} {'R²':>8}")
    print("  " + "-" * 46)
    for r in tr_v2:
        print(f"  {r['method']:<30} {r['mae']:8.2f} {r['r2']:8.4f}")

fs_path_v2 = os.path.join(CHECKPOINT_DIR, "fewshot_results.json")
if os.path.exists(fs_path_v2):
    with open(fs_path_v2) as f:
        fs_v2 = json.load(f)
    print("\nFew-Shot:")
    for r in fs_v2:
        print(f"  n={r['n_samples']:<6}: MAE = {r['mae_mean']:.2f} ± {r['mae_std']:.2f}")


## 8. Как запустить модель на ваших данных

### 8.1 Быстрый старт

Подготовьте CSV файл с колонками:
```
cement,water,sand,coarse_agg,fine_add_1,fine_add_2,plasticizer,age_days
350,180,750,1050,0,0,5,28
300,190,800,1000,50,0,8,7
```

Запустите:
```bash
python predict.py --input your_data.csv --checkpoint_dir experiments/checkpoints
```

Результат: CSV с колонками `predicted_strength`, `uncertainty_95ci`,
`lower_bound`, `upper_bound`.

### 8.2 Использование из Python

```python
from predict import predict_ensemble
import pandas as pd

df = pd.read_csv("your_data.csv")
mu, ci95 = predict_ensemble(df, "experiments/checkpoints")
print(f"Прочность: {mu[0]:.1f} ± {ci95[0]:.1f} МПа")
```

### 8.3 Параметры

| Параметр | Значение | Описание |
|----------|---------|----------|
| `--method single` | Лучшая модель | Быстрее, MAE ≈ 8.49 |
| `--method ensemble` | Все 16 моделей | Точнее, MAE ≈ 7.23 |
| `--mc_samples 30` | MC-Dropout сэмплы | Больше = точнее σ, медленнее |


In [ ]:
# ══════════════════════════════════════════════════════════════
# ЗАПУСК МОДЕЛИ НА ВАШИХ ДАННЫХ
# ══════════════════════════════════════════════════════════════
#
# Способ 1: Загрузите свой CSV через кнопку ниже
# Способ 2: Используйте встроенный тестовый набор
#
# Формат CSV:
#   cement, water, sand, coarse_agg, fine_add_1, fine_add_2,
#   plasticizer, age_days

import subprocess

INPUT_CSV = None

# --- Попытка загрузить файл через Colab виджет ---
try:
    from google.colab import files
    print(" Загрузите CSV файл с составами бетона:")
    print("   (или нажмите 'Cancel' чтобы использовать встроенный тест)\n")
    uploaded = files.upload()
    if uploaded:
        INPUT_CSV = list(uploaded.keys())[0]
        print(f"\n Загружен: {INPUT_CSV}")
except Exception:
    pass

# --- Fallback: встроенный тест ---
if not INPUT_CSV:
    INPUT_CSV = "test_input.csv"
    print(f"Используем встроенный тестовый набор: {INPUT_CSV}")

# --- Запуск predict.py ---
cmd = [
    sys.executable, "predict.py",
    "--input", INPUT_CSV,
    "--checkpoint_dir", CHECKPOINT_DIR,
    "--method", "ensemble",
    "--mc_samples", "30",
    "--output", "predictions_output.csv",
]
print(f"\n Запуск: {' '.join(cmd[-8:])}\n")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(" Ошибка:")
    print(result.stderr)
else:
    # --- Показать результаты ---
    out_df = pd.read_csv("predictions_output.csv")

    print("\n" + "═" * 70)
    print("РЕЗУЛЬТАТЫ ПРОГНОЗА")
    print("═" * 70)

    display_cols = ["cement", "water", "age_days",
                    "predicted_strength", "uncertainty_95ci",
                    "lower_bound", "upper_bound"]
    display_cols = [c for c in display_cols if c in out_df.columns]
    print(out_df[display_cols].to_string(index=False))
    print(f"\n Предсказания сохранены: predictions_output.csv")
    print(f"   Загрузить результат можно через Files → predictions_output.csv")


## 9. Заключение

### Ключевые результаты (исправленный пайплайн)

| Метрика | Best GAN (individual) | Ridge Stacking (16 моделей) |
|---------|:---------------------:|:---------------------------:|
| MAE | 8.49 МПа | **7.23 МПа** |
| R² | 0.53 | **0.70** |
| PICP (95%) | 94.6% | 92.2% |

### Что сработало

1. **GAN > Supervised** (−0.83 МПа): adversarial training с NEAT+BNN
   дискриминатором заставляет генератор выдавать более реалистичные
   предсказания (MAE: 9.32 → 8.49)
2. **Ridge Stacking** (−1.26 МПа): комбинация разнообразных моделей
   (разные lr, архитектуры, physics-informed) через мета-обучение
3. **Physics-informed модели в стекинге**: модели с physics loss
   индивидуально хуже (MAE~10), но добавили −0.49 МПа к стекингу
   благодаря разнообразию ошибок
4. **Feature interactions**: cement×log(age), wc×log(age) улучшили
   модель, добавив физически осмысленные признаки
5. **Pre-training обобщает**: transfer learning MAE = 5.18 без
   дообучения (модель работает на узких выборках)

### Что не сработало и почему

1. **Physics GAN (λ=0.3)**: слишком агрессивный физический штраф
   привёл к best_epoch=95 и MAE=9.93. Но эти модели полезны в стекинге!
2. **Transfer EWC/Replay**: pre-trained модель обобщает лучше,
   чем fine-tune на 144 сэмплах → catastrophic forgetting
3. **Монотонность 46%**: модель не гарантирует ∂f/∂t ≥ 0
4. **Multi-property (slump)**: 873/3745 записей с осадкой конуса
   → недостаточно данных для второго выхода

### Ошибка split и уроки

Первоначальный `stratified_split` давал оптимистичные MAE ≈ 5 МПа
из-за утечки данных. После исправления на `grouped_stratified_split`
MAE выросло до 8–9 МПа, но это **честная** оценка. Stacking позволил
снизить MAE до 7.23 МПа. Урок: валидационная стратегия критически важна
для реальной оценки модели.

### Воспроизводимость

- Фиксированный seed=42 для всех экспериментов
- Все чекпоинты сохранены в Google Drive
- Код: [github.com/Surmbruh/concrete-strength](https://github.com/Surmbruh/concrete-strength)


In [ ]:
print("=" * 60)
print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ (исправленный пайплайн)")
print("=" * 60)
print(f"\n{'Задача':<40} {'Результат':>15}")
print("-" * 55)
print(f"{'Supervised baseline (grouped split)':<40} {'MAE = 9.32':>15}")
print(f"{'Best GAN (individual)':<40} {'MAE = 8.49':>15}")
print(f"{'Ridge Stacking (16 моделей)':<40} {'MAE = 7.23':>15}")
print(f"{'Улучшение GAN vs Supervised':<40} {'-9.0%':>15}")
print(f"{'Transfer (no adaptation)':<40} {'MAE = 5.18':>15}")
print(f"{'Few-shot (n=50)':<40} {'MAE = 12.68':>15}")
print(f"{'Few-shot (n=2000)':<40} {'MAE = 9.86':>15}")
print(f"{'Time (t=28d)':<40} {'MAE = 6.89':>15}")
print(f"{'PICP (95% CI)':<40} {'92-98%':>15}")
